# Preprocesamiento para Modelado

Este notebook retoma el trabajo realizado en `Preprocesamiento_y_analisis.ipynb` y completa
los pasos pendientes para dejar el dataset en condiciones optimas para el modelado supervisado.
El objetivo central es predecir el nivel de adiccion al gaming (`addiction_level`) a partir de
variables conductuales, demograficas y fisicas del jugador, **sin incluir otras variables de
salud mental que podrian introducir fuga de datos**.

Las etapas cubiertas en este notebook son:

| # | Etapa | Descripcion |
|---|-------|-------------|
| 1 | Definicion del target | Conversion de `addiction_level` continuo a clases ordinales |
| 2 | Exclusion de leakage | Eliminar variables que revelan informacion del target |
| 3 | Analisis de correlacion | Identificar features con mayor poder predictivo |
| 4 | Escalado de features | Aplicar `StandardScaler` correctamente (fit en train) |
| 5 | Split train/test | Division estratificada para preservar balance de clases |
| 6 | Validacion y DQR | Verificar calidad del dataset final |
| 7 | Guardado | Exportar dataset listo para modelado |

## 0. Contexto y objetivos del notebook

### Punto de partida

El notebook `Preprocesamiento_y_analisis.ipynb` produjo el archivo `Dataset/dataset_preprocessed.csv`
con las siguientes caracteristicas:

| Aspecto | Detalle |
|---------|--------|
| **Filas** | 200,000 registros |
| **Columnas** | 45 (39 originales + 4 derivadas + 3 one-hot de gender) |
| **Valores nulos** | 0 (imputacion con mediana completada) |
| **Duplicados** | 0 |
| **Encoding** | `gender` ya esta en formato one-hot |
| **Outliers** | Winsorizados al percentil 1-99 en 6 variables |

### Que faltaba

A pesar del exhaustivo preprocesamiento previo, quedaban pendientes cuatro tareas criticas antes
de poder entrenar cualquier modelo:

1. **Definicion del target**: `addiction_level` es una variable continua (0-10). Para modelado
   de clasificacion se necesita convertirla a categorias interpretables.
2. **Exclusion de data leakage**: Varias variables del dataset son *co-outcomes* de salud mental
   (ansiedad, depresion, felicidad, soledad) que comparten causa raiz con la adiccion. Usarlas como
   features haria el modelo artificialmente bueno y no generalizable.
3. **Escalado de features**: Muchos algoritmos (regresion logistica, SVM, redes neuronales, KNN)
   son sensibles a la escala de las variables. Variables como `income` (0-150K) e `internet_quality`
   (1-10) no son comparables sin normalizar.
4. **Division train/test estratificada**: Necesaria para evaluar el modelo correctamente y evitar
   que el ajuste del scaler contamine el conjunto de prueba.

### Objetivos SMART de este notebook

| # | Objetivo | Criterio de exito |
|---|----------|------------------|
| O1 | Codificar el target en 3 clases (bajo/moderado/alto) | Clases definidas con umbral justificado, sin valores nulos |
| O2 | Eliminar variables con riesgo de data leakage | >= 8 features excluidas con justificacion documentada |
| O3 | Aplicar StandardScaler correctamente | Scaler ajustado *solo* en train; media ≈ 0 y std ≈ 1 en train |
| O4 | Generar split 80/20 estratificado | Diferencia de distribucion de clases entre train y test < 0.5pp |
| O5 | Exportar dataset final | Archivo `Dataset/data_preprocesada.csv` guardado y verificado |

## 1. Librerias y configuracion

Se importan las librerias necesarias y se definen los parametros configurables del notebook.
La parametrizacion facilita reproducir el experimento con distintos umbrales o proporciones
de split sin modificar el codigo principal.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# ── Parametros configurables ────────────────────────────────────────────────
PROCESSED_INPUT = Path("Dataset/dataset_preprocessed.csv")
OUTPUT_PATH     = Path("Dataset/data_preprocesada.csv")

# Umbrales para conversion del target a clases
LOW_THRESH  = 3.0   # [0, 3.0)  → bajo
HIGH_THRESH = 6.5   # [3.0, 6.5) → moderado  |  [6.5, 10] → alto

# Division train/test
TEST_SIZE    = 0.20
RANDOM_STATE = 42

SAVE_OUTPUT  = True

print("Librerias cargadas correctamente.")

## 2. Carga del dataset preprocesado

Se carga el archivo generado por el notebook anterior (`dataset_preprocessed.csv`). Este archivo
ya paso por limpieza de invalidos, winsorizacion, ingenieria de features, imputacion con mediana
y encoding one-hot de la variable `gender`. Es el punto de partida de este notebook.

In [ ]:
df = pd.read_csv(PROCESSED_INPUT)

print(f"Shape del dataset cargado: {df.shape}")
print(f"Columnas: {df.shape[1]}")
print(f"Registros: {df.shape[0]:,}")
df.head()

## 3. Revision del punto de partida

Antes de cualquier transformacion, se verifica el estado del dataset recibido: tipos de datos,
valores faltantes y conteo de valores unicos por columna. Esta revision confirma que el
preprocesamiento previo se aplico correctamente.

In [ ]:
overview = pd.DataFrame({
    "dtype":   df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique":  df.nunique(dropna=False),
}).sort_index()

print(f"Nulos totales: {df.isna().sum().sum()}")
print(f"Duplicados:    {df.duplicated().sum()}")
overview

## 4. Definicion y codificacion del target

La variable `addiction_level` es continua con rango 0-10. Para formular el problema como
clasificacion multiclase, se convierte a tres categorias ordinales: **bajo**, **moderado** y **alto**.
Esto permite que los modelos aprendan a distinguir perfiles de riesgo y que los resultados sean
interpretables para stakeholders no tecnicos.

### 4.1 Distribucion de `addiction_level`

Antes de definir los umbrales de corte, se analiza la distribucion de la variable original.
Esto permite elegir umbrales que sean a la vez estadisticamente razonables y clinicamente
interpretables, evitando clases demasiado desbalanceadas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Histograma ---
axes[0].hist(df["addiction_level"], bins=40, color="#f4a259", edgecolor="white")
axes[0].axvline(
    df["addiction_level"].mean(), color="#e63946", linewidth=2,
    label=f"Media: {df['addiction_level'].mean():.2f}"
)
axes[0].axvline(
    df["addiction_level"].median(), color="#457b9d", linewidth=2, linestyle="--",
    label=f"Mediana: {df['addiction_level'].median():.2f}"
)
axes[0].axvline(LOW_THRESH,  color="#2d6a4f", linewidth=1.5, linestyle=":", label=f"Corte bajo: {LOW_THRESH}")
axes[0].axvline(HIGH_THRESH, color="#6a4c93", linewidth=1.5, linestyle=":", label=f"Corte alto: {HIGH_THRESH}")
axes[0].set_title("Distribucion de addiction_level (con umbrales de corte)")
axes[0].set_xlabel("addiction_level")
axes[0].set_ylabel("Frecuencia")
axes[0].legend(fontsize=8)

# --- Percentiles ---
percentiles = [10, 25, 33, 50, 66, 75, 90, 95, 99]
values = [df["addiction_level"].quantile(p / 100) for p in percentiles]
axes[1].barh([f"p{p}" for p in percentiles], values, color="#577590")
axes[1].set_title("Percentiles de addiction_level")
axes[1].set_xlabel("Valor")
for i, v in enumerate(values):
    axes[1].text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print("Estadisticos descriptivos de addiction_level:")
print(df["addiction_level"].describe().round(3))

### 4.2 Creacion de clases: bajo / moderado / alto

Se definen tres rangos basados en la distribucion observada y en la interpretabilidad clinica:

| Clase | Rango | Interpretacion |
|-------|-------|----------------|
| **bajo** | [0.0, 3.0) | Uso recreativo sin indicadores de adiccion |
| **moderado** | [3.0, 6.5) | Patron de juego intenso con senales de riesgo |
| **alto** | [6.5, 10.0] | Comportamiento adictivo con impacto funcional |

Los umbrales `LOW_THRESH=3.0` y `HIGH_THRESH=6.5` se alinean con el percentil 50 y el
percentil ~90 respectivamente, generando clases con volumen suficiente para entrenar modelos.
Son parametros configurables al inicio del notebook.

In [ ]:
df["addiction_class"] = pd.cut(
    df["addiction_level"],
    bins=[-0.001, LOW_THRESH, HIGH_THRESH, 10.001],
    labels=["bajo", "moderado", "alto"]
)

class_counts = df["addiction_class"].value_counts().sort_index()
class_pct    = (class_counts / len(df) * 100).round(1)

class_summary = pd.DataFrame({
    "cantidad":     class_counts,
    "porcentaje_%": class_pct,
    "rango":        ["[0.0, 3.0)", "[3.0, 6.5)", "[6.5, 10.0]"]
})
print("Distribucion de clases del target:")
print(class_summary)

### 4.3 Analisis de balance de clases

Un desbalance severo entre clases puede sesgar los modelos hacia la clase mayoritaria.
Se visualiza la distribucion para identificar si sera necesario aplicar tecnicas de
rebalanceo (SMOTE, class_weight) durante el modelado.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors      = ["#4d908e", "#f4a259", "#e63946"]
class_order = ["bajo", "moderado", "alto"]
counts      = [class_counts[c] for c in class_order]

# --- Barras ---
bars = axes[0].bar(class_order, counts, color=colors, edgecolor="white", width=0.5)
axes[0].set_title("Distribucion de clases del target")
axes[0].set_xlabel("Clase de adiccion")
axes[0].set_ylabel("Cantidad de registros")
for bar, count in zip(bars, counts):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 400,
        f"{count:,}\n({count/len(df)*100:.1f}%)",
        ha="center", va="bottom", fontsize=9
    )

# --- Pie ---
axes[1].pie(
    counts, labels=class_order, colors=colors,
    autopct="%1.1f%%", startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
axes[1].set_title("Proporcion de clases")

plt.suptitle("Balance de clases del target: addiction_class", fontsize=12)
plt.tight_layout()
plt.show()

# Ratio de desbalance (clase mayoritaria / clase minoritaria)
ratio = max(counts) / min(counts)
print(f"Ratio de desbalance (mayor/menor): {ratio:.2f}x")
if ratio > 3:
    print("⚠ Desbalance significativo. Considerar class_weight='balanced' o SMOTE en modelado.")
else:
    print("Balance aceptable para la mayoria de algoritmos.")

## 5. Exclusion de variables con riesgo de data leakage

El **data leakage** ocurre cuando una feature contiene informacion que, en un contexto real,
no estaria disponible en el momento de la prediccion, o cuando la feature es en realidad
un *co-outcome* del target, no una causa.

En este dataset, las variables de salud mental como `anxiety_score`, `depression_score`,
`happiness_score` y similares **comparten la misma causa raiz que `addiction_level`**: son
consecuencias del mismo patron de comportamiento, no predictores independientes.
Incluirlas haria el modelo artificialmente preciso en entrenamiento, pero inutil en produccion
porque en la vida real no se miderian *antes* de saber si alguien tiene adiccion.

### 5.1 Variables excluidas y justificacion

A continuacion se documenta cada variable eliminada y el motivo especifico de su exclusion.

In [ ]:
LEAKAGE_FEATURES = [
    "anxiety_score",
    "depression_score",
    "mental_burden_index",
    "happiness_score",
    "loneliness_score",
    "aggression_score",
    "social_interaction_score",
    "relationship_satisfaction",
]

leakage_justification = {
    "anxiety_score":             "Outcome psicologico co-ocurrente con adiccion; no disponible antes del diagnostico",
    "depression_score":          "Outcome psicologico co-ocurrente; correlacion espuria con el target",
    "mental_burden_index":       "Variable derivada de anxiety + depression + stress: fuga directa del target",
    "happiness_score":           "Resultado afectivo de la adiccion, no su causa",
    "loneliness_score":          "Co-outcome: la adiccion y la soledad comparten causa, pero no son causalmente ordenadas",
    "aggression_score":          "Consecuencia conductual del patron adictivo",
    "social_interaction_score":  "Resultado del comportamiento de juego; no es un predictor causal independiente",
    "relationship_satisfaction": "Outcome afectado por el nivel de adiccion, no predictor previo",
}

leakage_df = pd.DataFrame.from_dict(
    leakage_justification, orient="index", columns=["Justificacion de exclusion"]
).rename_axis("Variable")

print(f"Total de features excluidas por leakage: {len(LEAKAGE_FEATURES)}")
leakage_df

### 5.2 Construccion del conjunto de features limpio

Una vez identificadas las variables a excluir, se construye el vector de features final.
Se eliminan las columnas con leakage y las dos columnas del target (original y codificada),
conservando unicamente las variables que un modelo podria usar en produccion.

In [ ]:
# Columnas que NO son features (target o leakage)
EXCLUDE_COLS = LEAKAGE_FEATURES + ["addiction_level", "addiction_class"]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f"Features seleccionadas: {len(feature_cols)} columnas")
print(f"Features excluidas:     {len(EXCLUDE_COLS)} columnas (leakage + target)")
print(f"Total original:         {df.shape[1]} columnas")
print()

# Clasificacion de las features finales
BINARY_COLS  = ["headset_usage", "gender_Female", "gender_Male", "gender_Other"]
cols_to_scale = [c for c in feature_cols if c not in BINARY_COLS]

print("Features continuas / ordinales (se escalaran):")
for i, col in enumerate(cols_to_scale, 1):
    print(f"  {i:2d}. {col}")
print()
print("Features binarias (NO se escalaran):")
for col in BINARY_COLS:
    print(f"   - {col}")

## 6. Analisis de correlacion con el target

Se calcula la correlacion de Pearson en valor absoluto entre cada feature y `addiction_level`
(usando la version continua para mayor sensibilidad numerica). Este analisis permite:

- Identificar las variables con mayor poder predictivo.
- Verificar que la exclusion de features de leakage no elimino variables con correlacion real.
- Documentar el umbral de correlacion minima que se usara como criterio de calidad (KPI).

In [ ]:
corr_with_target = (
    df[feature_cols + ["addiction_level"]]
    .corr(numeric_only=True)["addiction_level"]
    .drop("addiction_level")
    .abs()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 11))
bar_colors = ["#e63946" if v >= 0.10 else "#457b9d" for v in corr_with_target.values]

ax.barh(
    corr_with_target.index[::-1],
    corr_with_target.values[::-1],
    color=bar_colors[::-1]
)
ax.axvline(0.10, color="#e63946", linewidth=1.5, linestyle="--", label="Umbral r = 0.10")
ax.set_title("Correlacion absoluta de Pearson con addiction_level\n(solo features sin leakage)")
ax.set_xlabel("Correlacion absoluta de Pearson")
ax.legend()
plt.tight_layout()
plt.show()

above_threshold = corr_with_target[corr_with_target >= 0.10]
print(f"Features con correlacion >= 0.10: {len(above_threshold)}")
print(above_threshold.round(4).to_string())

### 6.1 Distribucion de las top features por clase de adiccion

Para las 6 variables con mayor correlacion con el target, se compara su distribucion
entre las tres clases. Los boxplots permiten visualizar si hay separacion real entre
grupos, lo cual indica que la feature sera util para el modelo de clasificacion.

In [ ]:
top_features = corr_with_target.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
class_order  = ["bajo", "moderado", "alto"]
box_colors   = ["#4d908e", "#f4a259", "#e63946"]

for ax, feature in zip(axes.flat, top_features):
    data_by_class = [
        df[df["addiction_class"] == cls][feature].dropna()
        for cls in class_order
    ]
    bp = ax.boxplot(
        data_by_class,
        patch_artist=True,
        labels=class_order,
        medianprops=dict(color="white", linewidth=2),
        flierprops=dict(marker=".", markersize=2, alpha=0.3)
    )
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
    ax.set_title(feature, fontsize=9, fontweight="bold")
    ax.set_ylabel("Valor")
    ax.set_xlabel("Clase de adiccion")

plt.suptitle("Top 6 features por separacion entre clases de addiction_class", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 7. Escalado de features

El escalado es indispensable para algoritmos que miden distancias o usan gradientes: regresion
logistica, SVM, KNN, redes neuronales, Ridge/Lasso. Sin escalado, una variable como `income`
(rango 5K-150K) domina completamente sobre `sleep_hours` (rango 0-24) en terminos de magnitud,
aunque ambas sean igualmente relevantes.

Se usa `StandardScaler` (z-score), que transforma cada feature a media=0, std=1:

$$z = \frac{x - \mu_{train}}{\sigma_{train}}$$

> **Regla critica**: El scaler se ajusta (`fit`) **solo** en el conjunto de entrenamiento y
> luego se aplica (`transform`) a ambos conjuntos. Ajustarlo sobre toda la data introduciria
> fuga de informacion del conjunto de prueba hacia el proceso de normalizacion.

### 7.1 Division train / test estratificada

Se divide el dataset antes de ajustar el scaler para garantizar que el conjunto de prueba
sea completamente independiente. La estratificacion asegura que la proporcion de clases del
target sea similar en ambas particiones, lo cual es especialmente importante cuando hay
desbalance entre clases.

In [ ]:
X = df[feature_cols].copy()
y = df["addiction_class"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]:,} registros ({(1-TEST_SIZE)*100:.0f}%)")
print(f"Conjunto de prueba:        {X_test.shape[0]:,} registros ({TEST_SIZE*100:.0f}%)")
print()

dist_comparison = pd.DataFrame({
    "train_n":  y_train.value_counts().sort_index(),
    "train_%":  (y_train.value_counts(normalize=True) * 100).sort_index().round(2),
    "test_n":   y_test.value_counts().sort_index(),
    "test_%":   (y_test.value_counts(normalize=True) * 100).sort_index().round(2),
})
print("Distribucion de clases por particion:")
print(dist_comparison)

# Verificar que la estratificacion fue correcta (diff < 0.5 pp)
max_diff = (dist_comparison["train_%"] - dist_comparison["test_%"]).abs().max()
print(f"\nDiferencia maxima entre train% y test%: {max_diff:.2f} pp")
if max_diff < 0.5:
    print("Estratificacion correcta: diferencia < 0.5 pp en todas las clases.")

### 7.2 Ajuste y aplicacion del StandardScaler

Se escalan las variables continuas y ordinales. Las variables binarias (indicadores 0/1 como
`headset_usage` y los dummies de `gender`) se conservan en su escala original porque ya estan
en el rango [0, 1] y el escalado no aportaria informacion adicional.

In [ ]:
scaler = StandardScaler()

# Copias para no modificar los DataFrames originales
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

# fit SOLO en train → transform en ambos (evita leakage del test)
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

print(f"Columnas escaladas:       {len(cols_to_scale)}")
print(f"Columnas sin escalar:     {len(BINARY_COLS)} (binarias)")
print()

# Verificacion: media y std en train deben ser ~0 y ~1
scale_check = pd.DataFrame({
    "media_train":  X_train_scaled[cols_to_scale].mean().round(4),
    "std_train":    X_train_scaled[cols_to_scale].std().round(4),
    "media_test":   X_test_scaled[cols_to_scale].mean().round(4),
    "std_test":     X_test_scaled[cols_to_scale].std().round(4),
})

print("Estadisticos post-escalado (primeras 8 features):")
print(scale_check.head(8))
print()
print("Resumen: media y std de train deben ser ~0 y ~1")
print(scale_check[["media_train", "std_train"]].describe().round(4))

## 8. Validacion del dataset final

Antes de exportar, se realiza una verificacion completa del estado del dataset:
completitud, shapes, balance de clases y coherencia de los datos escalados.
Esta validacion actua como control de calidad final del pipeline.

In [ ]:
print("=" * 58)
print("VALIDACION DEL DATASET FINAL")
print("=" * 58)

print(f"\n[1] SHAPES")
print(f"    X_train: {X_train_scaled.shape}  |  y_train: {y_train.shape}")
print(f"    X_test:  {X_test_scaled.shape}   |  y_test:  {y_test.shape}")

print(f"\n[2] COMPLETITUD")
print(f"    Nulos en X_train: {X_train_scaled.isna().sum().sum()}")
print(f"    Nulos en X_test:  {X_test_scaled.isna().sum().sum()}")
print(f"    Nulos en y_train: {y_train.isna().sum()}")
print(f"    Nulos en y_test:  {y_test.isna().sum()}")

print(f"\n[3] BALANCE DE CLASES")
class_dist = pd.DataFrame({
    "train_n":   y_train.value_counts().sort_index(),
    "train_%":   (y_train.value_counts(normalize=True) * 100).sort_index().round(1),
    "test_n":    y_test.value_counts().sort_index(),
    "test_%":    (y_test.value_counts(normalize=True) * 100).sort_index().round(1),
})
print(class_dist)

print(f"\n[4] ESCALADO — Rango de medias y desv. std en train (esperado: ~0 y ~1)")
train_stats = X_train_scaled[cols_to_scale].agg(["mean", "std"])
print(f"    Media:  min={train_stats.loc['mean'].min():.4f}, max={train_stats.loc['mean'].max():.4f}")
print(f"    Std:    min={train_stats.loc['std'].min():.4f},  max={train_stats.loc['std'].max():.4f}")

print(f"\n[5] TIPOS DE DATOS")
print(f"    dtypes X_train (unicos): {X_train_scaled.dtypes.unique()}")
print(f"    dtype y_train: {y_train.dtype}")

print(f"\n[6] VARIABLES BINARIAS (sin escalar — rango debe ser [0, 1])")
for col in BINARY_COLS:
    mn = X_train_scaled[col].min()
    mx = X_train_scaled[col].max()
    print(f"    {col}: min={mn}, max={mx}")

print("=" * 58)

## 9. Data Quality Report (DQR) — Etapa de preparacion para modelado

El Data Quality Report documenta el estado del dataset al final de este notebook, cubriendo
las dimensiones de calidad relevantes para la etapa de modelado: completitud, exclusion de
leakage, configuracion del escalado y balance del target.

In [ ]:
print("=" * 63)
print("DATA QUALITY REPORT — PREPARACION PARA MODELADO")
print("=" * 63)

print(f"\n[1] COMPLETITUD")
print(f"    Registros totales:         {len(df):,}")
print(f"    Features seleccionadas:    {len(feature_cols)} columnas")
print(f"    Nulos en features (X):     {X.isna().sum().sum()}")
print(f"    Nulos en target (y):       {y.isna().sum()}")

print(f"\n[2] EXCLUSION DE DATA LEAKAGE")
print(f"    Variables excluidas:       {len(LEAKAGE_FEATURES)}")
for var in LEAKAGE_FEATURES:
    print(f"    - {var}")

print(f"\n[3] CODIFICACION DEL TARGET")
print(f"    Variable original:         addiction_level (continua 0–10)")
print(f"    Variable codificada:       addiction_class (ordinal 3 clases)")
print(f"    Umbrales:")
print(f"      bajo:      [0.0,  {LOW_THRESH})")
print(f"      moderado:  [{LOW_THRESH}, {HIGH_THRESH})")
print(f"      alto:      [{HIGH_THRESH}, 10.0]")
print(f"    Distribucion:")
for cls in ["bajo", "moderado", "alto"]:
    n   = int((y == cls).sum())
    pct = n / len(y) * 100
    print(f"      {cls:10s}: {n:,} ({pct:.1f}%)")

print(f"\n[4] PARTICION TRAIN / TEST")
print(f"    Proporcion test:           {TEST_SIZE:.0%}")
print(f"    Random state:              {RANDOM_STATE}")
print(f"    Estratificacion:           si (por addiction_class)")
print(f"    Registros train:           {len(X_train):,}")
print(f"    Registros test:            {len(X_test):,}")

print(f"\n[5] ESCALADO")
print(f"    Metodo:                    StandardScaler (z-score)")
print(f"    Ajustado en:               solo train (no hay leakage de test)")
print(f"    Columnas escaladas:        {len(cols_to_scale)}")
print(f"    Columnas sin escalar:      {len(BINARY_COLS)} (binarias)")

print(f"\n[6] RIESGOS Y CONSIDERACIONES PARA MODELADO")
print("    - Desbalance de clases: la clase 'alto' es la menos frecuente.")
print("      Considerar class_weight='balanced' o SMOTE si el modelo favorece 'bajo'.")
print("    - Dataset sintetico: validar generalizacion con datos reales antes de produccion.")
print("    - Variables ordinales (stress_level, etc.) escaladas como continuas.")
print("    - El scaler esta ajustado en train: re-ajustar si se cambia el split.")
print("=" * 63)

## 10. Arquitectura del pipeline actualizado

El siguiente diagrama extiende el pipeline documentado en el notebook anterior, agregando
las etapas de preparacion para modelado completadas en este notebook.

```mermaid
flowchart TD
    A[("Fuente\nDataset sintetico\n10M x 40 features")] --> B

    subgraph PREV["Notebook anterior — Preprocesamiento_y_analisis.ipynb"]
        B["Carga y validacion"] --> C["Limpieza y outliers"]
        C --> D["Feature engineering\ngaming_intensity_index\nonline_social_ratio\ngaming_screen_share\nmental_burden_index"]
        D --> E["Imputacion mediana\n+ one-hot gender"]
        E --> F[("dataset_preprocessed.csv\n200K x 45 features")]
    end

    F --> G

    subgraph ESTE["Este notebook — preprocesamiento.ipynb"]
        G["Carga dataset preprocesado"] --> H
        H["Codificacion del target\naddiction_level → bajo/moderado/alto"] --> I
        I["Exclusion de leakage\n8 variables de salud mental eliminadas"] --> J
        J["Analisis de correlacion\ncon el target"] --> K
        K["Split estratificado\n80%% train / 20%% test"] --> L
        L["StandardScaler\nfit en train, transform en ambos"] --> M
        M["Validacion y DQR"] --> N
        N[("data_preprocesada.csv\ntrain+test escalados\n36 features + target")]
    end

    N --> O

    subgraph MODELADO["Tarea siguiente — Modelado"]
        O["Cargar data_preprocesada.csv"]
        O --> P["Entrenamiento de modelos"]
        P --> Q["Evaluacion y seleccion"]
    end

    style PREV fill:#f0f4f8,stroke:#457b9d
    style ESTE fill:#e8f5e9,stroke:#2d6a4f
    style MODELADO fill:#f5f5f5,stroke:#999
```

### Parametros del pipeline reproducible

| Parametro | Valor | Descripcion |
|-----------|-------|-------------|
| `LOW_THRESH` | 3.0 | Umbral bajo/moderado para addiction_class |
| `HIGH_THRESH` | 6.5 | Umbral moderado/alto para addiction_class |
| `TEST_SIZE` | 0.20 | Proporcion del conjunto de prueba |
| `RANDOM_STATE` | 42 | Semilla para reproducibilidad |
| `scaler` | StandardScaler | Normalizacion z-score, ajustada solo en train |

## 11. Guardado del dataset final

Se exporta un unico archivo `data_preprocesada.csv` que contiene:

| Columna | Descripcion |
|---------|-------------|
| 36 features | Variables seleccionadas, escaladas (StandardScaler) |
| `addiction_class` | Target codificado: bajo / moderado / alto |
| `addiction_level_original` | Valor continuo original (referencia) |
| `partition` | Indica si el registro pertenece a train o test |

La columna `partition` permite al notebook de modelado cargar directamente los subconjuntos
ya escalados sin necesidad de rehacer el split, garantizando reproducibilidad exacta.

```python
# Ejemplo de uso en el notebook de modelado:
df = pd.read_csv('Dataset/data_preprocesada.csv')
feat = [c for c in df.columns if c not in ['addiction_class','addiction_level_original','partition']]
X_train = df[df.partition == 'train'][feat]
X_test  = df[df.partition == 'test'][feat]
y_train = df[df.partition == 'train']['addiction_class']
y_test  = df[df.partition == 'test']['addiction_class']
```

In [ ]:
if SAVE_OUTPUT:
    # Reconstruir DataFrames con columnas adicionales de referencia
    df_train_out = X_train_scaled.copy()
    df_train_out["addiction_class"]         = y_train.values
    df_train_out["addiction_level_original"] = df.loc[X_train.index, "addiction_level"].values
    df_train_out["partition"]                = "train"

    df_test_out = X_test_scaled.copy()
    df_test_out["addiction_class"]         = y_test.values
    df_test_out["addiction_level_original"] = df.loc[X_test.index, "addiction_level"].values
    df_test_out["partition"]                = "test"

    df_export = pd.concat([df_train_out, df_test_out]).sort_index()

    df_export.to_csv(OUTPUT_PATH, index=False)

    print(f"Dataset guardado en: {OUTPUT_PATH.resolve()}")
    print(f"Shape final:         {df_export.shape}")
    print(f"Columnas:            {df_export.shape[1]}")
    print(f"Registros train:     {(df_export.partition == 'train').sum():,}")
    print(f"Registros test:      {(df_export.partition == 'test').sum():,}")
    print()
    print("Columnas del archivo exportado:")
    print(df_export.columns.tolist())
else:
    print("Guardado desactivado (SAVE_OUTPUT = False)")

## 12. Conclusiones y checklist para modelado

Este notebook completo los pasos de preprocesamiento pendientes orientados al modelado supervisado.
A continuacion se resume el estado final del dataset y se provee un checklist de lo que se
debe tener en cuenta al comenzar la etapa de modelado.

### Estado final del dataset

| Aspecto | Estado |
|---------|--------|
| **Registros** | 200,000 (160K train / 40K test) |
| **Features** | 36 (sin leakage, escaladas) |
| **Target** | `addiction_class`: bajo / moderado / alto |
| **Valores nulos** | 0 |
| **Escalado** | StandardScaler, ajustado en train |
| **Split** | 80/20 estratificado, random_state=42 |
| **Archivo output** | `Dataset/data_preprocesada.csv` |

### Checklist para la etapa de modelado

- [ ] Cargar `Dataset/data_preprocesada.csv` y separar por columna `partition`
- [ ] Verificar balance de clases y decidir si usar `class_weight='balanced'`
- [ ] Entrenar linea base: `DummyClassifier` y `LogisticRegression`
- [ ] Probar algoritmos de arboles: `RandomForest`, `GradientBoosting` o `XGBoost`
- [ ] Usar `StratifiedKFold` en cross-validation sobre el conjunto de train
- [ ] Evaluar con metricas apropiadas para multiclase: `f1_weighted`, `confusion_matrix`
- [ ] Analizar importancia de features para interpretar el modelo
- [ ] Reportar resultados finales sobre `X_test` (solo al final, no durante tuning)

### Consideraciones especiales

**Sobre el desbalance de clases**: La clase `alto` representa la minoria. Si el modelo
ignora estas predicciones, usar `class_weight='balanced'` en algoritmos que lo soporten,
o aplicar `SMOTE` sobre el conjunto de train antes del entrenamiento.

**Sobre el target**: `addiction_class` es una variable ordinal (bajo < moderado < alto).
Algoritmos de clasificacion ordinales o el encoding `{bajo: 0, moderado: 1, alto: 2}`
aprovechan mejor esta estructura que el one-hot encoding.